In [1]:
from llm_model.call_model import ModelCaller

In [4]:
#model_name = "openai/gpt-4.1"
model_name="gpt-4o-mini"
llm = ModelCaller(model_name , max_tokens=500)
#print(llm.call_model("Tell me about Deep space?"))

/Users/suryaatul/PythonWorkspace/AgenticAI/Notebooks/llm_model/call_model.py:23: ExperimentalWarning: AzureAIChatCompletionsModel is currently in preview and is subject to change. This preview is provided without a service-level agreement, and we don't recommend it for production workloads. Certain features might not be supported or might have constrained capabilities. For more information, see https://azure.microsoft.com/support/legal/preview-supplemental-terms
  model = AzureAIChatCompletionsModel(


### Use of @Tool

@tool is correct when

1. Tool has 1–2 simple arguments
2. Arguments are strings / primitives
3. You’re prototyping
4. You don’t need strict validation

When you have some complex schema with the function it is recommended to use StructuredTool , as it can be used for schema validations

In [45]:
#from langchain.tools import tool
from langchain_core.tools import   tool
import re

@tool
def sum_numbers_from_text(inputs: str) -> float:
    """
    Adds a list of numbers provided in the input string.
    
    Args:
        text: A string containing numbers that should be extracted and summed.
        
    Returns:
        The sum of all numbers found in the input.
    """
    # Use regular expressions to extract all numbers from the input
    numbers = [int(num) for num in re.findall(r'\d+', inputs)]
    result = sum(numbers)
    return result

In [207]:
#from langgraph.prebuilt import create_react_agent
from langchain.agents import create_agent 

agent_exec = create_agent(model=llm.get_model(), tools=[sum_numbers_from_text])
msgs = agent_exec.invoke({"messages": [("human", "Add the numbers -10, -20, -30")]})

In [208]:
print(msgs['messages'][-1].content)

The sum of the numbers -10, -20, and -30 is 60.


In [209]:
print(msgs['messages'][1].usage_metadata)

{'input_tokens': 88, 'output_tokens': 24, 'total_tokens': 112}


In [210]:
type(msgs['messages'][1].usage_metadata)

dict

### -------------------------------------------------------------------------------------------------------------------------

### Use of Structured_Tool

When you have some complex schema with the function it is recommended to use StructuredTool , as it can be used for schema validations

1. Multiple parameters
2. Optional parameters
3. Azure AI (strict schemas)
4. Production systems
5. You want predictable tool calls

In [296]:
import re
from langchain_core.tools import StructuredTool , tool

def sum_numbers_from_text2(inputs: str , average :bool , absolute:bool) -> float:
    """
    Adds a list of numbers provided in the input string , and calcuate average if flag is set to true.
    
    Args:
        text: A string containing numbers that should be extracted and summed.
        boolean : A boolean value in True or False to calculate the average of extracted and summed value
        boolean : boolean flag to incidicate of the value to return should be absolute
        
    Returns:
        The sum of all numbers found in the input if boolean flag is false
        the average of the numbers in the input if boolean flag is true
    """
    # Use regular expressions to extract all numbers from the input
    numbers = [int(num) for num in re.findall(r'\d+', inputs)]
  
    try:

        if not numbers:
            raise ValueError("No numbers in the input string.")
        else:
            result = sum(numbers)

        if absolute:
            result = abs(result)
        
        if average == False:
            return result
        else:
            return result/len(numbers)
    except Exception as e:
        print(f"Error caught as {e}")

In [297]:
from pydantic import BaseModel,Field


class SumInput(BaseModel):
    inputs: str=Field(description="Input String")
    average: bool = Field(description = " boolean value to calculate average of numbers passed")
    absolute: bool = Field(description = " boolean flag to indicate if value should be absolute")



In [298]:
from langchain_core.tools import StructuredTool


# sum_tool = StructuredTool.from_function(
#     "sum_numbers_from_text2",
#     args_schema=SumInput
#     #prompt="You are helpful assistant in performing calculations by extracting numbers from input string"
# )


tool = StructuredTool.from_function(
    func=sum_numbers_from_text2,
    name="addition_tool", # Define name explicitly
    description="Useful for identifying numeric values from string and sum them ", # Define description
    args_schema=SumInput
)

In [299]:
from langchain_core.messages import HumanMessage , AIMessage
from langchain.agents import create_agent 

agent_exec = create_agent(model=llm.get_model(), tools=[tool])
# msgs = agent_exec.invoke({"inputs": [("human", "Add the numbers -10, -20, -30")]})


msgs = agent_exec.invoke({
    "messages": [
        AIMessage(content="you are wonderful assistant to per calculation for numbers in input string"),
        HumanMessage(content="Add the numbers -10, -20, -30 only")
    ]
})


In [300]:
print(msgs['messages'])

[AIMessage(content='you are wonderful assistant to per calculation for numbers in input string', additional_kwargs={}, response_metadata={}, id='58b1848a-9836-43ea-ba54-093ef06f867e', tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='Add the numbers -10, -20, -30 only', additional_kwargs={}, response_metadata={}, id='7ea17daa-8c3b-46ad-aeb5-f4979e2a5649'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gpt-4o-mini-2024-07-18', 'token_usage': {'input_tokens': 107, 'output_tokens': 31, 'total_tokens': 138}, 'finish_reason': 'tool_calls'}, id='lc_run--019c140f-7822-7c92-a548-45b6fbb5c916-0', tool_calls=[{'name': 'addition_tool', 'args': {'absolute': False, 'average': False, 'inputs': '-10, -20, -30'}, 'id': 'call_JLE8swbRKw0Rz5PMStIVGNKD', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 107, 'output_tokens': 31, 'total_tokens': 138}), ToolMessage(content='60', name='addition_tool', id='93f3bc13-09a4-425b-ae59-467c1f2b0e5f'

In [301]:
print(msgs['messages'][-1].content)

The sum of the numbers -10, -20, and -30 is 60.


### ---------- Check for average calculation and params passed to tool ---------------

In [302]:
msgs = agent_exec.invoke({
    "messages": [
        AIMessage(content="you are wonderful assistant to per calculation for numbers in input string"),
        HumanMessage(content="Add the number -10, -20, -30 and calculate average")
    ]
})

In [303]:
print(msgs['messages'])

[AIMessage(content='you are wonderful assistant to per calculation for numbers in input string', additional_kwargs={}, response_metadata={}, id='58dd1c68-5601-4012-92fc-c75de10d96b4', tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='Add the number -10, -20, -30 and calculate average', additional_kwargs={}, response_metadata={}, id='49fe8e5f-6de6-47b6-a676-80d249c582f8'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gpt-4o-mini-2024-07-18', 'token_usage': {'input_tokens': 109, 'output_tokens': 31, 'total_tokens': 140}, 'finish_reason': 'tool_calls'}, id='lc_run--019c140f-9104-7561-bcf8-b5b5ee30820a-0', tool_calls=[{'name': 'addition_tool', 'args': {'absolute': False, 'average': True, 'inputs': '-10, -20, -30'}, 'id': 'call_mgT1qYJ6mwhw4zzS7sIVczf5', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 109, 'output_tokens': 31, 'total_tokens': 140}), ToolMessage(content='20.0', name='addition_tool', id='ea4dad06-a5c3-48af-8

In [304]:
print((msgs['messages'][2].tool_calls))

[{'name': 'addition_tool', 'args': {'absolute': False, 'average': True, 'inputs': '-10, -20, -30'}, 'id': 'call_mgT1qYJ6mwhw4zzS7sIVczf5', 'type': 'tool_call'}]


In [305]:
print(msgs['messages'][-1].content)

The average of the numbers -10, -20, and -30 is 20.0.


### ---------- Check for average calculation , absolute and params passed to tool ---------------

In [306]:
msgs = agent_exec.invoke({
    "messages": [
        AIMessage(content="you are wonderful assistant to per calculation for numbers in input string"),
        HumanMessage(content="calculate average -10, -20, -30 . Value has to be absolute number")
    ]
})

In [307]:
# Check for the tool arguments passed here
#tool_calls=[{'name': 'addition_tool', 'args': {'absolute': True, 'average': True, 'inputs': '-10, -20, -30'}

print(msgs['messages'])

[AIMessage(content='you are wonderful assistant to per calculation for numbers in input string', additional_kwargs={}, response_metadata={}, id='6007402f-6819-4fe1-9123-378c74325c32', tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='calculate average -10, -20, -30 . Value has to be absolute number', additional_kwargs={}, response_metadata={}, id='8a65fc21-a972-4460-bec5-f1854941af92'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gpt-4o-mini-2024-07-18', 'token_usage': {'input_tokens': 112, 'output_tokens': 31, 'total_tokens': 143}, 'finish_reason': 'tool_calls'}, id='lc_run--019c140f-b674-7261-87d0-8eb01752048e-0', tool_calls=[{'name': 'addition_tool', 'args': {'absolute': True, 'average': True, 'inputs': '-10, -20, -30'}, 'id': 'call_iAkBEIsZ88d5276LPSunSzm2', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 112, 'output_tokens': 31, 'total_tokens': 143}), ToolMessage(content='20.0', name='addition_tool', id='09983f

In [308]:
print((msgs['messages'][2].tool_calls))

[{'name': 'addition_tool', 'args': {'absolute': True, 'average': True, 'inputs': '-10, -20, -30'}, 'id': 'call_iAkBEIsZ88d5276LPSunSzm2', 'type': 'tool_call'}]


In [309]:
print(msgs['messages'][-1].content)

The average of the absolute values of -10, -20, and -30 is 20.0.


### ---------- Check Exception messages ---------------

In [310]:
msgs = agent_exec.invoke({
    "messages": [
        AIMessage(content="you are wonderful assistant to per calculation for numbers in input string"),
        HumanMessage(content=" Add the numbers ??"),
        HumanMessage(content="In case of error strictly return a json {{ 'result' : 'failed' }} ")
    ]
})

Error caught as No numbers in the input string.


In [311]:
print(msgs['messages'])

[AIMessage(content='you are wonderful assistant to per calculation for numbers in input string', additional_kwargs={}, response_metadata={}, id='0a5859ce-18cc-4c07-a340-b196c1e78578', tool_calls=[], invalid_tool_calls=[]), HumanMessage(content=' Add the numbers ??', additional_kwargs={}, response_metadata={}, id='ca91f39c-0915-48b9-8ac8-e1638f65b6a8'), HumanMessage(content="In case of error strictly return a json {{ 'result' : 'failed' }} ", additional_kwargs={}, response_metadata={}, id='e57325be-1871-4a19-81e1-a0bcd47760dc'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gpt-4o-mini-2024-07-18', 'token_usage': {'input_tokens': 121, 'output_tokens': 24, 'total_tokens': 145}, 'finish_reason': 'tool_calls'}, id='lc_run--019c140f-eea8-7500-a744-205abd2a70f6-0', tool_calls=[{'name': 'addition_tool', 'args': {'absolute': False, 'average': False, 'inputs': '??'}, 'id': 'call_A6yPuFmFPKJNpqrWPsO4ZVjF', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'

In [312]:
print(msgs['messages'][-1].content)

{"result":"failed"}
